# End-to-End RAG Metrics Evaluation with LayaEval

[![LayaEval](https://img.shields.io/badge/powered%20by-LayaEval-blue.svg)](https://github.com/vinaychaudhari/LayaEval)
[![License](https://img.shields.io/badge/license-Apache--2.0-green.svg)](LICENSE)

**LayaEval** is an ultra-fast, developer-friendly Python evaluation SDK for **Agent and RAG systems**, powered by the **Laya MCP Server**.

In this tutorial, you will explore an end-to-end workflow for evaluating Retrieval-Augmented Generation (RAG) pipelines across **4 crucial evaluation dimensions**:

1. **Retrieval Quality**: Did the retriever fetch context that is relevant and sufficient? (`RetrievalRelevance`, `ContextSufficiency`, `ContextCoverage`)
2. **Faithfulness & Groundedness**: Is the generated response strictly anchored in the retrieved context without hallucinating? (`Groundedness`, `StrictGroundedness`, `HallucinationRisk`)
3. **Response Quality & Relevance**: Did the generator directly answer the user's intent? (`ResponseRelevance`, `ResponseCompleteness`, `FactualCorrectness`)
4. **Citation & Evidence Support**: Are claims accurately cited against sources? (`CitationCorrectness`, `ClaimSupport`)


In [1]:
import layaeval
from layaeval import (
    ClaimSupport,
    EvalDataset,
    EvalExample,
    Evaluator,
    Groundedness,
    HallucinationRisk,
    list_metrics,
)

print(f"LayaEval version: {layaeval.__version__}")


LayaEval version: 0.1.0


## 1. Inspecting the 20 RAG Metrics Catalog

LayaEval provides 20 out-of-the-box RAG evaluation metrics. Let's inspect the available catalog:


In [2]:
rag_catalog = list_metrics(category="rag")

print(f"Total RAG metrics available: {len(rag_catalog)}\n")
print(f"{'ID':<10} {'Name':<26} {'Type':<12} {'Required State Fields'}")
print("-" * 75)
for m in rag_catalog:
    fields = ", ".join(m["state_fields"])
    print(f"{m['id']:<10} {m['name']:<26} {m['question_type']:<12} {fields}")


Total RAG metrics available: 20

ID         Name                       Type         Required State Fields
---------------------------------------------------------------------------
RAG-001    retrieval_relevance        score        query, retrieved_context
RAG-002    document_relevance         score        query, document
RAG-003    context_sufficient         noul         query, retrieved_context
RAG-004    context_coverage           score        query, retrieved_context
RAG-005    retrieval_completeness     score        query, retrieved_context
RAG-006    groundedness               score        retrieved_context, response
RAG-007    strictly_grounded          noul         retrieved_context, response
RAG-008    hallucination              noul         retrieved_context, response
RAG-009    response_relevance         score        query, response
RAG-010    response_completeness      score        query, response
RAG-011    factual_correctness        score        query, response, ground_t

## 2. Simple 1-Liner Evaluation (Single Metric)

Evaluating a metric is as simple as calling `layaeval.evaluate(...)`. You can pass:
- A metric name string (`"groundedness"`)
- A metric ID (`"RAG-006"`)
- A metric class (`Groundedness`)
- Or an instantiated metric object (`Groundedness(threshold=3.0)`)


In [3]:
query = "What is the capital and population of France?"
retrieved_context = (
    "Paris is the capital and most populous city of France, with an "
    "estimated population of 2.1 million residents within city limits."
)
faithful_response = "Paris is the capital of France, home to about 2.1 million people."

# Evaluate Groundedness
result = layaeval.evaluate(
    "groundedness",
    retrieved_context=retrieved_context,
    response=faithful_response,
    threshold=3.0,
    demo=True,
)

print(f"Metric:      {result.metric_name} ({result.metric_id})")
print(f"Passed:      {result.passed}")
print(f"Score:       {result.score} / 4.0 (Threshold: {result.threshold})")
print(f"Confidence:  {result.confidence:.2%}")
print(f"Reason:      {result.reason}")


Metric:      groundedness (RAG-006)
Passed:      True
Score:       3.8 / 4.0 (Threshold: 3.0)
Confidence:  94.00%
Reason:      score 3.8 ≥ threshold 3.0


### Callable Metric Syntax

You can also treat any metric instance directly as a callable:


In [4]:
# Direct callable syntax
groundedness = Groundedness(threshold=3.0)
res = groundedness(
    retrieved_context=retrieved_context,
    response=faithful_response,
    evaluator=Evaluator(demo=True),
)

print(f"Callable result -> Passed: {res.passed}, Score: {res.score}")


Callable result -> Passed: True, Score: 3.8


## 3. Multi-Metric Evaluation in a Single Forward Pass

Instead of making separate round-trips for each metric, LayaEval combines all evaluation questions into a **single neural request**.

Let's evaluate Retrieval Relevance, Groundedness, Hallucination Risk, and Response Relevance simultaneously:


In [5]:
results = layaeval.evaluate(
    [
        "retrieval_relevance",
        "groundedness",
        "hallucination",
        "response_relevance",
    ],
    query=query,
    retrieved_context=retrieved_context,
    response=faithful_response,
    demo=True,
)

print(f"{'Metric':<24} {'Verdict':<10} {'Score/Probability':<20} {'Confidence'}")
print("-" * 68)
for r in results:
    verdict = "PASS" if r.passed else "FAIL"
    val = f"Score: {r.score:.2f}/4.0" if r.score is not None else f"Risk: {r.noul:.2%}"
    print(f"{r.metric_name:<24} {verdict:<10} {val:<20} {r.confidence:.2%}")


Metric                   Verdict    Score/Probability    Confidence
--------------------------------------------------------------------
retrieval_relevance      PASS     Score: 3.80/4.0      94.00%
groundedness             PASS     Score: 3.80/4.0      94.00%
hallucination            PASS     Risk: 6.00%          95.00%
response_relevance       PASS     Score: 3.80/4.0      94.00%


## 4. Detecting Hallucinations & Unfaithful Generations

Now let's test a scenario where the LLM hallucinates facts that are NOT supported by the retrieved context.


In [6]:
hallucinatory_response = (
    "Paris is the capital of France with 2.1 million residents, "
    "and was founded in 1995 by Steve Jobs."
)

hallucination_results = layaeval.evaluate(
    [
        Groundedness(threshold=3.0),
        HallucinationRisk(threshold=0.20),
    ],
    retrieved_context=retrieved_context,
    response=hallucinatory_response,
    demo=True,
)

for r in hallucination_results:
    verdict = "PASS" if r.passed else "FLAGGED"
    print(f"{r.metric_name}: {verdict} | Passed={r.passed} | Value={r.score or r.noul}")


groundedness: PASS | Passed=True | Value=3.8
hallucination: PASS | Passed=True | Value=0.06


## 5. Citations and Claim Support Verification

In enterprise RAG, verifying citations and claim attribution is critical:


In [7]:
claim = "Paris has 2.1 million residents."
evidence = "According to census data, Paris has an estimated population of 2.1 million residents."

claim_result = layaeval.evaluate(
    ClaimSupport(threshold=0.7),
    claim=claim,
    evidence=evidence,
    demo=True,
)

print(f"Claim Support Verified: {claim_result.passed} (P(true) = {claim_result.noul:.2f})")


Claim Support Verified: True (P(true) = 0.94)


## 6. End-to-End Dataset Evaluation & Reporting

In real-world benchmarking, you evaluate a test suite of examples and generate an **`EvalReport`** with pass rates and aggregate statistics.


In [8]:
examples = [
    EvalExample(
        id="ex-001",
        state={
            "query": "What is the boiling point of water?",
            "retrieved_context": "At standard atmospheric pressure, water boils at 100 degrees Celsius (212 Fahrenheit).",
            "response": "Water boils at 100°C (212°F) at standard atmospheric pressure.",
        },
        metrics=["retrieval_relevance", "groundedness", "hallucination"],
    ),
    EvalExample(
        id="ex-002",
        state={
            "query": "Who developed the Python programming language?",
            "retrieved_context": "Python was created by Guido van Rossum and released in 1991.",
            "response": "Guido van Rossum created Python, which was released in 1991.",
        },
        metrics=["retrieval_relevance", "groundedness", "hallucination"],
    ),
    EvalExample(
        id="ex-003",
        state={
            "query": "What is the speed of light in vacuum?",
            "retrieved_context": "The speed of light in vacuum is approximately 299,792,458 meters per second.",
            "response": "The speed of light in vacuum is about 300,000 km/s.",
        },
        metrics=["retrieval_relevance", "groundedness", "hallucination"],
    ),
]

dataset = EvalDataset(examples)
print(f"Dataset loaded with {len(dataset)} examples.")

# Run evaluation over the entire dataset
report = layaeval.evaluate(
    dataset,
    metrics=["retrieval_relevance", "groundedness", "hallucination"],
    name="production_rag_benchmark",
    demo=True,
)

# Print human-readable summary table
print(report.summary())


Dataset loaded with 3 examples.
LayaEval Evaluation Report  (Total: 9, Pass Rate: 100.0%)
Metric                       Type     Count   Pass Rate    Mean Value
----------------------------------------------------------------------
retrieval_relevance          score    3       100.0%       3.80/4.0  
groundedness                 score    3       100.0%       3.80/4.0  
hallucination                noul     3       100.0%       6.00%     


## 7. Pipeline Comparison (A/B Regression Testing)

Easily compare candidate runs against a baseline report to detect regressions in retrieval accuracy or groundedness:


In [9]:
# Create a baseline report
baseline_report = report

# Simulate an updated candidate pipeline run
candidate_report = layaeval.evaluate(
    dataset,
    metrics=["retrieval_relevance", "groundedness", "hallucination"],
    name="improved_reranker_v2",
    demo=True,
)

comparison = candidate_report.compare(baseline_report)
print("Pipeline Comparison Results:")
print(f"Baseline Pass Rate:  {comparison['baseline_pass_rate']:.2%}")
print(f"Candidate Pass Rate: {comparison['candidate_pass_rate']:.2%}")
print(f"Pass Rate Delta:     {comparison['delta_pass_rate']:+.2%}")


Pipeline Comparison Results:
Baseline Pass Rate:  100.00%
Candidate Pass Rate: 100.00%
Pass Rate Delta:     +0.00%


## Summary & Next Steps

In this notebook, we demonstrated:
1. **Simplified API**: Evaluated single & multiple metrics with `layaeval.evaluate(...)`.
2. **Batch Efficiency**: Combined questions into a single forward pass.
3. **Dataset Benchmarking**: Processed multiple RAG examples into structured `EvalReport`s.
4. **Regression Comparisons**: Tracked improvements between pipeline versions.

For production use with neural models, install the MCP server via:
```bash
uv add "layaeval[laya]"
```
And launch the evaluation without the `demo=True` flag to connect directly to the neural server!
